In [1]:
# =========================================================
# STAGE 0: IMPORTS AND BASIC SETUP
# =========================================================
import os
import re
import numpy as np
import torch
import transformers

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
)
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Optional: disable wandb logging
os.environ["WANDB_DISABLED"] = "true"

# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [2]:
# =========================================================
# STAGE 1: LOAD DATASET
# =========================================================
# Load dataset from Hugging Face Hub
dataset = load_dataset("ziq/ingredient_to_sugar_level")

print("Original train columns:", dataset["train"].column_names)

# Adjust these column names based on your dataset structure
train_columns = set(dataset["train"].column_names)

text_column = "ingredients_clean" if "ingredients_clean" in train_columns else "ingredients"
label_column = "sugar_per_serving_z" if "sugar_per_serving_z" in train_columns else "sugar"

# Rename columns into standard names
dataset = dataset.rename_column(text_column, "text")
dataset = dataset.rename_column(label_column, "labels")

# Keep only needed columns
keep_columns = {"text", "labels"}
remove_columns = [c for c in dataset["train"].column_names if c not in keep_columns]
if remove_columns:
    dataset = dataset.remove_columns(remove_columns)

print(dataset)
print("Example sample:", dataset["train"][0])

README.md:   0%|          | 0.00/527 [00:00<?, ?B/s]

data/train-00000-of-00001-92fba141b71e21(…):   0%|          | 0.00/5.22M [00:00<?, ?B/s]

data/test-00000-of-00001-1cb825f6efe55c9(…):   0%|          | 0.00/457k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/22592 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1965 [00:00<?, ? examples/s]

Original train columns: ['src', 'ingredients', 'sugar']
DatasetDict({
    train: Dataset({
        features: ['text', 'labels'],
        num_rows: 22592
    })
    test: Dataset({
        features: ['text', 'labels'],
        num_rows: 1965
    })
})
Example sample: {'text': '1 pound ground beef, 1 tablespoon dehydrated onion, 1 teaspoon salt, 1/2 teaspoon pepper, 1 teaspoon garlic powder, 1/4 teaspoon oregano, 1 cup uncooked penne pasta, 1 cup water, 1 (15-ounce) can fire-roasted diced tomatoes, 1/4 cup Parmesan cheese, 1/2 cup shredded mozzarella cheese, 4-6 fresh basil leaves, torn', 'labels': -0.7369838161043166}


In [3]:
# =========================================================
# STAGE 2: PREPROCESSING
# =========================================================
import re
import numpy as np

# ---------------------------------------------------------
# STEP 2.1: Define text cleaning function
# ---------------------------------------------------------
# This function cleans the ingredient text to reduce noise.
# It removes:
# - URLs
# - fractions and numbers
# - common measurement units
# - special characters
# It keeps meaningful ingredient words for the model.

def clean_ingredient_text(text):
    if text is None:
        return ""

    # 1. Convert to lowercase
    text = text.lower()

    # 2. Remove URLs
    text = re.sub(r"http\S+|www\S+", "", text)

    # 3. Remove fractions like 1/2, 3/4
    text = re.sub(r"\b\d+/\d+\b", " ", text)

    # 4. Remove decimal numbers like 1.5
    text = re.sub(r"\b\d+\.\d+\b", " ", text)

    # 5. Remove integers like 1, 2, 10
    text = re.sub(r"\b\d+\b", " ", text)

    # 6. Remove common measurement units
    units_pattern = (
        r"\b(cup|cups|tbsp|tablespoon|tablespoons|tsp|teaspoon|teaspoons|"
        r"oz|ounce|ounces|lb|pound|pounds|g|gram|grams|kg|ml|l|liter|liters|"
        r"pinch|clove|cloves|slice|slices|can|cans|package|packages)\b"
    )
    text = re.sub(units_pattern, " ", text)

    # 7. Remove special characters, keep letters, commas and spaces
    text = re.sub(r"[^a-z,\s]", " ", text)

    # 8. Normalize repeated spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text


# ---------------------------------------------------------
# STEP 2.2: Apply text cleaning
# ---------------------------------------------------------
# Create a new cleaned text column
def preprocess_example(example):
    example["clean_text"] = clean_ingredient_text(example["text"])
    return example

dataset = dataset.map(preprocess_example)

print("Before cleaning:")
print(dataset["train"][0]["text"])
print("\nAfter cleaning:")
print(dataset["train"][0]["clean_text"])


# ---------------------------------------------------------
# STEP 2.3: Remove empty / too-short samples
# ---------------------------------------------------------
def is_valid_example(example):
    return example["clean_text"] is not None and len(example["clean_text"].strip()) > 3

dataset = dataset.filter(is_valid_example)


# ---------------------------------------------------------
# STEP 2.4: Clip extreme label outliers
# ---------------------------------------------------------
# This helps stabilize regression training
train_labels = np.array(dataset["train"]["labels"])
lower_bound = np.percentile(train_labels, 1)
upper_bound = np.percentile(train_labels, 99)

def clip_label(example):
    example["labels"] = float(np.clip(example["labels"], lower_bound, upper_bound))
    return example

dataset = dataset.map(clip_label)

print("\nLabel clipping range:")
print("Lower bound:", lower_bound)
print("Upper bound:", upper_bound)


# ---------------------------------------------------------
# STEP 2.5: Prepare raw train/test splits after preprocessing
# ---------------------------------------------------------
train_dataset_raw = dataset["train"].shuffle(seed=42)
test_dataset_raw = dataset["test"].shuffle(seed=42)

print("\nTrain size:", len(train_dataset_raw))
print("Test size:", len(test_dataset_raw))

Map:   0%|          | 0/22592 [00:00<?, ? examples/s]

Map:   0%|          | 0/1965 [00:00<?, ? examples/s]

Before cleaning:
1 pound ground beef, 1 tablespoon dehydrated onion, 1 teaspoon salt, 1/2 teaspoon pepper, 1 teaspoon garlic powder, 1/4 teaspoon oregano, 1 cup uncooked penne pasta, 1 cup water, 1 (15-ounce) can fire-roasted diced tomatoes, 1/4 cup Parmesan cheese, 1/2 cup shredded mozzarella cheese, 4-6 fresh basil leaves, torn

After cleaning:
ground beef, dehydrated onion, salt, pepper, garlic powder, oregano, uncooked penne pasta, water, fire roasted diced tomatoes, parmesan cheese, shredded mozzarella cheese, fresh basil leaves, torn


Filter:   0%|          | 0/22592 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1965 [00:00<?, ? examples/s]

Map:   0%|          | 0/22592 [00:00<?, ? examples/s]

Map:   0%|          | 0/1965 [00:00<?, ? examples/s]


Label clipping range:
Lower bound: -0.8095738329613784
Upper bound: 4.374588149270094

Train size: 22592
Test size: 1965


In [4]:
# =========================================================
# STAGE 2: PREPROCESSING- NO TOUCH DATASET
# =========================================================
# We use one tokenizer for each model separately later,
# so here we only prepare the raw train/test split.

# train_dataset_raw = dataset["train"].shuffle(seed=42)
# test_dataset_raw = dataset["test"].shuffle(seed=42)

# print("Train size:", len(train_dataset_raw))
# print("Test size:", len(test_dataset_raw))

In [5]:
# =========================================================
# STAGE 3: HELPER FUNCTIONS
# =========================================================
# Regression metrics because your current setup predicts a continuous sugar score.
def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    # Flatten predictions
    preds = np.squeeze(predictions)
    labels = np.squeeze(labels)

    # Calculate metrics
    mae = mean_absolute_error(labels, preds)
    mse = mean_squared_error(labels, preds)
    rmse = np.sqrt(mse)
    r2 = r2_score(labels, preds)

    return {
        "mae": float(mae),
        "rmse": float(rmse),
        "r2": float(r2),
    }


def tokenize_dataset(dataset_dict, tokenizer, max_length=128):
    """
    Tokenize dataset using the provided tokenizer.
    """
    def tokenize_function(examples):
        return tokenizer(
            examples["clean_text"],
            padding="max_length",
            truncation=True,
            max_length=max_length,
        )

    tokenized_train = dataset_dict["train"].map(tokenize_function, batched=True)
    tokenized_test = dataset_dict["test"].map(tokenize_function, batched=True)

    tokenized_train = tokenized_train.with_format("torch")
    tokenized_test = tokenized_test.with_format("torch")

    return tokenized_train, tokenized_test


def build_trainer(model_name, output_dir, train_dataset, test_dataset):
    """
    Build tokenizer, model, training args, and trainer for one model.
    """
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    # Re-tokenize using this model's tokenizer
    dataset_dict = {
        "train": train_dataset_raw,
        "test": test_dataset_raw,
    }
    tokenized_train, tokenized_test = tokenize_dataset(dataset_dict, tokenizer)

    # Load model for regression
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=1
    )
    model.config.problem_type = "regression"
    model.to(device)

    training_args = TrainingArguments(
        output_dir=output_dir,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=8,
        warmup_steps=100,
        max_steps=3000,              # change if needed, default 2000
        learning_rate=2e-5,
        fp16=torch.cuda.is_available(),
        logging_steps=20,
        eval_strategy="steps",
        save_strategy="steps",
        eval_steps=100,
        save_steps=100,
        save_total_limit=3,
        load_best_model_at_end=True,
        push_to_hub=False,
        report_to="none",
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_train,
        eval_dataset=tokenized_test,
        # tokenizer=tokenizer,
        compute_metrics=compute_metrics,
    )

    return trainer, tokenizer, tokenized_test

In [6]:
# =========================================================
# STAGE 4: TRAIN DISTILBERT
# =========================================================
distilbert_name = "distilbert-base-uncased"
distilbert_output = "distilbert_sugar_model"

distilbert_trainer, distilbert_tokenizer, distilbert_test_dataset = build_trainer(
    model_name=distilbert_name,
    output_dir=distilbert_output,
    train_dataset=train_dataset_raw,
    test_dataset=test_dataset_raw,
)

# Train DistilBERT
distilbert_trainer.train()

# Save final DistilBERT model
distilbert_trainer.save_model(distilbert_output)
distilbert_tokenizer.save_pretrained(distilbert_output)

# Evaluate DistilBERT
distilbert_results = distilbert_trainer.evaluate()
print("DistilBERT Results:", distilbert_results)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/22592 [00:00<?, ? examples/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


DistilBERT Results: {'eval_loss': 1.0034279823303223, 'eval_mae': 0.44145670533180237, 'eval_rmse': 0.7083144793151651, 'eval_r2': 0.45476746559143066, 'eval_runtime': 8.2503, 'eval_samples_per_second': 238.175, 'eval_steps_per_second': 14.909, 'epoch': 8.498583569405099}


In [7]:
# =========================================================
# STAGE 5: TRAIN BERT
# =========================================================
bert_name = "bert-base-uncased"
bert_output = "bert_sugar_model"

bert_trainer, bert_tokenizer, bert_test_dataset = build_trainer(
    model_name=bert_name,
    output_dir=bert_output,
    train_dataset=train_dataset_raw,
    test_dataset=test_dataset_raw,
)

# Train BERT
bert_trainer.train()

# Save final BERT model
bert_trainer.save_model(bert_output)
bert_tokenizer.save_pretrained(bert_output)

# Evaluate BERT
bert_results = bert_trainer.evaluate()
print("BERT Results:", bert_results)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/22592 [00:00<?, ? examples/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


BERT Results: {'eval_loss': 1.0164437294006348, 'eval_mae': 0.45470842719078064, 'eval_rmse': 0.7128988187136143, 'eval_r2': 0.44768697023391724, 'eval_runtime': 15.0734, 'eval_samples_per_second': 130.362, 'eval_steps_per_second': 8.16, 'epoch': 8.498583569405099}


In [8]:
# =========================================================
# STAGE 6: COMPARE BOTH MODELS
# =========================================================
print("\n========== MODEL COMPARISON ==========")
print("DistilBERT MAE :", distilbert_results.get("eval_mae"))
print("DistilBERT RMSE:", distilbert_results.get("eval_rmse"))
print("DistilBERT R2:", distilbert_results.get("eval_r2"))

print("BERT MAE       :", bert_results.get("eval_mae"))
print("BERT RMSE      :", bert_results.get("eval_rmse"))
print("BERT R2      :", bert_results.get("eval_r2"))


========== MODEL COMPARISON ==========
DistilBERT MAE : 0.44145670533180237
DistilBERT RMSE: 0.7083144793151651
DistilBERT R2: 0.45476746559143066
BERT MAE       : 0.45470842719078064
BERT RMSE      : 0.7128988187136143
BERT R2      : 0.44768697023391724


In [9]:
# =========================================================
# STAGE 7: TEST BOTH MODELS ON ONE SAMPLE
# =========================================================
sample_text = "1 cup sugar, 2 tablespoons honey, 1 cup milk, 1 teaspoon vanilla extract"

def predict_single_text(model, tokenizer, text):
    """
    Run inference for one text input.
    """
    model.eval()

    encoded = tokenizer(
        text,
        padding="max_length",
        truncation=True,
        max_length=128,
        return_tensors="pt"
    )

    input_ids = encoded["input_ids"].to(device)
    attention_mask = encoded["attention_mask"].to(device)

    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        prediction = outputs.logits.squeeze().item()

    return prediction


distilbert_pred = predict_single_text(
    distilbert_trainer.model,
    distilbert_tokenizer,
    sample_text
)

bert_pred = predict_single_text(
    bert_trainer.model,
    bert_tokenizer,
    sample_text
)

print("\n========== SAMPLE PREDICTION ==========")
print("Input text:", sample_text)
print("DistilBERT predicted sugar score:", distilbert_pred)
print("BERT predicted sugar score      :", bert_pred)


========== SAMPLE PREDICTION ==========
Input text: 1 cup sugar, 2 tablespoons honey, 1 cup milk, 1 teaspoon vanilla extract
DistilBERT predicted sugar score: 1.810668706893921
BERT predicted sugar score      : 2.2943387031555176
